In [ ]:
// The fundamental split, computed by overload resolution: an lvalue argument
// binds the T& overload, so the T&& overload only ever catches rvalues
// (a prvalue or an xvalue).
template <typename T> const char* category(T&)  { return "lvalue"; }
template <typename T> const char* category(T&&) { return "rvalue"; }
// Among rvalues: an xvalue's decltype is a reference (T&&), a prvalue's is not (T).
template <typename T> const char* rvalue_kind() {
    return std::is_rvalue_reference_v<T> ? "xvalue" : "prvalue";
}
/** A type that logs which constructor ran, so a copy and a move are visible. */
struct ValueCatDemo {
    int n;
    ValueCatDemo() : n(0) {}
    ValueCatDemo(int i) : n(i) {}
    ValueCatDemo(const ValueCatDemo& o) : n(o.n) { cout << format("  copy constructor (from n={})\n", o.n); }
    ValueCatDemo(ValueCatDemo&& o) noexcept : n(o.n) { cout << format("  move constructor (from n={})\n", o.n); o.n = 0; }
};
{
    // ---------- Part A: the fundamental lvalue / rvalue split ----------
    int demo_a = 1;                 // a named, addressable object
    std::string s = "hello";
    cout << format("{:22} {}\n", "demo_a",           category(demo_a));
    cout << format("{:22} {}\n", "s",                category(s));
    cout << format("{:22} {}\n", "s[0]",             category(s[0]));
    cout << format("{:22} {}\n", "demo_a + 1",       category(demo_a + 1));
    cout << format("{:22} {}\n", "a new std::string",category(std::string("world")));
    cout << format("{:22} {}\n", "s.size()",         category(s.size()));
    cout << format("{:22} {}\n", "std::move(demo_a)",category(std::move(demo_a)));
    cout << format("{:22} {}\n", "std::move(s)",     category(std::move(s)));

    // ---------- Part B: rvalues split into prvalue and xvalue ----------
    cout << format("std::move(demo_a) is a {}\n", rvalue_kind<decltype(std::move(demo_a))>());
    cout << format("a new std::string is a {}\n", rvalue_kind<decltype(std::string("world"))>());
    cout << format("demo_a + 1 is a {}\n",        rvalue_kind<decltype(demo_a + 1)>());

    // ---------- Part C: the ++ asymmetry (ex:pre_vs_post) ----------
    int c = 0, d = 0;
    cout << format("++c : {}  (prefix ++ returns a reference to c)\n", category(++c));
    cout << format("d++ : {}  (postfix ++ returns a copy)\n",          category(d++));
    // The consequence: only an lvalue may sit on the left of '='.
    ++c = 1;       // prefix ++ yields an lvalue: assignment goes through
    // d++ = 1;    // compile error: "lvalue required as left operand of assignment"
    cout << format("after (++c)=1: c={} d={} (d++ = 1 does not compile)\n", c, d);

    // ---------- Part D: the payoff -- an xvalue enables a move ----------
    cout << "p (prvalue in)  / q (from xvalue) / r (from lvalue):\n";
    ValueCatDemo p = ValueCatDemo(3);   // a prvalue, materialized directly in p
    ValueCatDemo q = std::move(p);      // p becomes an xvalue -> move, p is drained
    ValueCatDemo r = q;                 // q is an lvalue -> copy
    cout << format("p={} q={} r={} (p was emptied by the move)\n", p.n, q.n, r.n);
}